# H&M Fashion Data: Data Preparation

This notebook creates clean data by fixing the issues found in the data overview notebook. It handles missing and inconsistent values, converts the date column to datetime, encodes customer IDs as integers, turns duplicate transactions into a `quantity` column, and reduces memory usage by downcasting data types. The cleaned tables are saved as Parquet files in the `data/processed/` folder so that they can be used by the following notebooks for analysis.

In [1]:
import pandas as pd
import numpy as np
import os

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.4f}".format)

RAW_DIR = "../data/"
PROCESSED_DIR = "../data/processed/"

## Load Data

In [2]:
articles = pd.read_parquet(RAW_DIR + "articles.parquet")
customers = pd.read_parquet(RAW_DIR + "customers.parquet")
transactions = pd.read_parquet(RAW_DIR + "transactions_train.parquet")

In [3]:
def memory_gb(df):
    """
Return the memory usage of a DataFrame in GB.

Parameters
df : DataFrame
    The table to measure.

Returns
float
    Memory usage in GB, rounded to 2 decimals.
"""

    total_memory_gb = df.memory_usage(deep=True).sum() / (1024**3)
    return round(total_memory_gb, 2)


In [4]:
articles_mem_before = memory_gb(articles)
customers_mem_before = memory_gb(customers)
transactions_mem_before = memory_gb(transactions)

print(f"articles {articles_mem_before} GB\ncustomers {customers_mem_before} GB\ntransactions {transactions_mem_before} GB")


articles 0.05 GB
customers 0.25 GB
transactions 3.37 GB


## 2. Clean Customers

In [6]:
customers["FN"] = customers["FN"].fillna(0).astype('int8')
customers["Active"] = customers["Active"].fillna(0).astype('int8')
print(customers["FN"].value_counts(dropna=False))
print(customers["Active"].value_counts(dropna=False))
print(customers.dtypes)


FN
0    895050
1    476930
Name: count, dtype: int64
Active
0    907576
1    464404
Name: count, dtype: int64
customer_id                   str
FN                           int8
Active                       int8
club_member_status            str
fashion_news_frequency        str
age                       float64
postal_code                   str
dtype: object


In [12]:
customers["fashion_news_frequency"] = customers["fashion_news_frequency"].replace({"None":"NONE"}).fillna("NONE")
customers["fashion_news_frequency"].value_counts()

fashion_news_frequency
NONE         893722
Regularly    477416
Monthly         842
Name: count, dtype: int64

### Postal Code

The most frequent postal code appears for 120,303 customers (about 8.8%), while the second most frequent one appears for only 261. This value may be a default placeholder, but this could not be verified because postal codes are hashed. Since none of the planned analyses depend on postal codes, the column is left unchanged. The value is stored in `placeholder_postal_code` in case it needs to be handled later.

In [18]:
customers["postal_code"].value_counts()
placeholder_postal_code = (customers["postal_code"].value_counts()).index[0]
placeholder_postal_code


'2c29ae653a9282cce4151bd87643c907644e09541abc28ae87dea0d1f6603b1c'

In [25]:
customers["age"].max() #99.0
customers["age"].isnull().sum() #15861
customers["age"] = customers["age"].astype("Int8")
assert customers["age"].isnull().sum() == 15861
customers["age"].head()


0    49
1    25
2    24
3    54
4    52
Name: age, dtype: Int8

In [27]:
customers["club_member_status"].isnull().sum()
customers["club_member_status"] = customers["club_member_status"].fillna("UNKNOWN")
customers["club_member_status"] = customers["club_member_status"].astype("category")
customers["fashion_news_frequency"] = customers["fashion_news_frequency"].astype("category")
customers["club_member_status"].value_counts()

club_member_status
ACTIVE        1272491
PRE-CREATE      92960
UNKNOWN          6062
LEFT CLUB         467
Name: count, dtype: int64

In [30]:
customers["customer_idx"] = np.arange(len(customers), dtype="int32")
print(customers["customer_idx"].min())
print(customers["customer_idx"].max())
print(customers["customer_idx"].dtype)
customers["customer_idx"].head()

0
1371979
int32


0    0
1    1
2    2
3    3
4    4
Name: customer_idx, dtype: int32